In [1]:
d = {}
d[(1, 2)] = 'точка А'
print(d)

{(1, 2): 'точка А'}


In [2]:
try:
    d[[1, 2]] = 'точка В'
except TypeError as e:
    print('TypeError: ', e)

TypeError:  cannot use 'list' as a dict key (unhashable type: 'list')


In [4]:
x = 42
print(type(globals()))
print(globals()['x'])

<class 'dict'>
42


In [8]:
class User:
    pass

u = User()
u.name = 'Anna'
print(u.__dict__)

{'name': 'Anna'}


In [13]:
a = {'Anna': 5, 'Bob': 4}
b = dict(Anna=5, Bob=4)
c = dict(zip(['Anna', 'Bob'], [5, 4]))
e = {name: len(name) for name in ['Anna', 'Bob']}


In [16]:
groups = dict.fromkeys(['A', 'B', 'C'], [])
groups['A'].append('Anna')
groups

{'A': ['Anna'], 'B': ['Anna'], 'C': ['Anna']}

In [17]:
groups = {g: [] for g in ['A', 'B', 'C']}
groups['A'].append('Anna')
groups

{'A': ['Anna'], 'B': [], 'C': []}

### Доступы [], get, in

In [25]:
stock = {'apple': 100, 'pear': 0}
print(stock.get('banana'))
print(stock.get('banana', 0))
print('apple' in stock, 100 in stock, 100 in stock.values())

try:
    stock['banana']
except KeyError as e:
    print('KeyError:', e)

None
0
True False True
KeyError: 'banana'


### Группировку

In [26]:
words = ['apple', 'avocado', 'banana', 'bluberry', 'cherry']
res = {}

for w in words:
    if w[0] not in res:
        res[w[0]] = []
    res[w[0]].append(w)
print(res)

{'a': ['apple', 'avocado'], 'b': ['banana', 'bluberry'], 'c': ['cherry']}


In [27]:
res = {}
for w in words:
    res.setdefault(w[0], []).append(w)
print(res)

{'a': ['apple', 'avocado'], 'b': ['banana', 'bluberry'], 'c': ['cherry']}


In [28]:
from collections import defaultdict, Counter

In [30]:
res = defaultdict(list)

for w in words:
    res[w[0]].append(w)
print(res)

print(res['d'])
print('d' in res)

defaultdict(<class 'list'>, {'a': ['apple', 'avocado'], 'b': ['banana', 'bluberry'], 'c': ['cherry']})
[]
True


### Порядок, объединение, view

In [31]:
d1 = {'a': 1, 'b': 2}
d2 = {'b': 2, 'a': 1}
print(d1 == d2, list(d1), list(d2))

True ['a', 'b'] ['b', 'a']


In [34]:
default = {'theme': 'light', 'lang': 'ru'}
user = {'theme': 'dark'}
print(default | user)
print({**default, **user})

{'theme': 'dark', 'lang': 'ru'}
{'theme': 'dark', 'lang': 'ru'}


In [39]:
cfg = {'host': 'localhost', 'port': 8080}
keys = cfg.keys()
print(keys)

cfg['debug'] = True
print(keys)

req = {'host', 'port', 'user'}
print('не хватает:', req - cfg.keys())

dict_keys(['host', 'port'])
dict_keys(['host', 'port', 'debug'])
не хватает: {'user'}


In [42]:
prices = {'apple': 100, 'pear': 0, 'plum': 0, 'kiwi': 50}
try:
    for k in prices:
        if prices[k] ==0:
            del prices[k]
except RuntimeError as e:
    print('RuntimeError: ', e)

RuntimeError:  dictionary changed size during iteration


In [46]:
prices = {'apple': 100, 'pear': 0, 'plum': 0, 'kiwi': 50}
print({k: v for k, v in prices.items() if v != 0})
print(prices.items())

{'apple': 100, 'kiwi': 50}
dict_items([('apple', 100), ('pear', 0), ('plum', 0), ('kiwi', 50)])


### Хеш-таблица

In [47]:
hash(-1) == -2

True

In [48]:
print(hash(42), hash(-1), hash(-2))

42 -2 -2


In [57]:
print(hash('hello'))

-8468631885584244416


In [53]:
print(hash((1, 2)))

-3550055125485641917


In [51]:
hash((1, [2, 3]))

TypeError: unhashable type: 'list'

In [58]:
N = 2**61 - 1

In [59]:
print(hash(N), hash(0))

0 0


In [60]:
d = {-1: 'minus 1', -2: 'minus 2'}

In [61]:
d[-1]

'minus 1'

In [62]:
class Key:
    def __init__(self, name, h):
        self.name = name
        self.h = h

    def __hash__(self):
        print(f'   hash({self.name}) -> {self.h}')
        return self.h

    def __eq__(self, other):
        print(f'   __eq__: {self.name} == {other.name}?')
        return self.name == other.name

    def __repr__(self):
        return self.name

In [65]:
d = {}
print('put А')
d[Key('A', 1)] = 1
print('put B')
d[Key('B', 2)] = 2
print('put C')
d[Key('C', 1)] = 3


print('search C')
print(d[Key('C', 1)])

put А
   hash(A) -> 1
put B
   hash(B) -> 2
put C
   hash(C) -> 1
   __eq__: A == C?
search C
   hash(C) -> 1
   __eq__: A == C?
   __eq__: C == C?
3


In [73]:
import random, string

def rnd():
    return ''.join(random.choices(string.ascii_lowercase, k=6))

for size in (8, 64, 1024):
    seen = {}
    while True:
        w = rnd()
        i = hash(w) % size
        if i in seen and seen[i] != w:
            print(f'size={size}: {seen[i]!r} и {w!r} -> ячейка {i}, после {len(seen) + 1} строк')
            break
        seen[i] = w

size=8: 'oclllw' и 'jhvljp' -> ячейка 7, после 5 строк
size=64: 'dpsabs' и 'jbjryi' -> ячейка 33, после 14 строк
size=1024: 'wfoerf' и 'snefvo' -> ячейка 657, после 31 строк


#### Рост словаря и компактное хранение

In [3]:
import sys

d = {}

prev = sys.getsizeof(d)
print(f'len=0, bytes={prev}')
for i in range(100):
    d[i] = i
    cur = sys.getsizeof(d)
    if cur != prev:
        print(f'len={i + 1}, bytes={cur} --- таблица перестроена')
        prev = cur

len=0, bytes=64
len=1, bytes=224 --- таблица перестроена
len=6, bytes=352 --- таблица перестроена
len=11, bytes=632 --- таблица перестроена
len=22, bytes=1168 --- таблица перестроена
len=43, bytes=2264 --- таблица перестроена
len=86, bytes=4688 --- таблица перестроена


In [ ]:
# a == b hash(a) == hash(b)
# 1, 1.0, True

In [4]:
print(1 == 1.0 == True, hash(1), hash(1.0), hash(True))

True 1 1 1


In [5]:
d = {
    1: 'int',
    1.0: 'float',
    True: 'bool',
}
print(d)

{1: 'bool'}


In [6]:
d[1] = 'int'
d[2] = 'int'
print(d)

{1: 'int', 2: 'int'}


In [10]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

A = Point(1, 2)
B = Point(1, 2)
print(A == B)
print(Point.__hash__)

try:
    {Point(1, 2)}
except TypeError as e:
    print(e)

True
None
cannot use 'Point' as a set element (unhashable type: 'Point')


In [12]:
class Point:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

a = {Point(1, 2)}
a

{<__main__.Point at 0x10a527a10>}

In [21]:
p = Point(1, 2)
visited = {p}
p.x = 100

In [22]:
print(p in visited)
print(p in list(visited))

False
True


In [24]:
import time

class GoodKey:
    def __init__(self, v):
        self.v = v

    def __hash__(self):
        return hash(self.v)

    def __eq__(self, other):
        return self.v == other.v

class BadKey(GoodKey):
    def __hash__(self):
        return 42

for cls in (GoodKey, BadKey):
    start = time.perf_counter()
    s = {cls(i) for i in range(5000)}
    print(f'{cls.__name__}: {time.perf_counter() - start:.4f} сек')

GoodKey: 0.0020 сек
BadKey: 0.4144 сек


In [74]:
GoodKey.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 3,
              '__init__': <function __main__.GoodKey.__init__(self, v)>,
              '__hash__': <function __main__.GoodKey.__hash__(self)>,
              '__eq__': <function __main__.GoodKey.__eq__(self, other)>,
              '__static_attributes__': ('v',),
              '__dict__': <attribute '__dict__' of 'GoodKey' objects>,
              '__weakref__': <attribute '__weakref__' of 'GoodKey' objects>,
              '__doc__': None})

In [25]:
try:
    {{1, 2}}
except TypeError as e:
    print(e)

cannot use 'set' as a set element (unhashable type: 'set')


In [26]:
m = set()
type(m)

set

In [28]:
nm = frozenset((1, 2))
{nm}

{frozenset({1, 2})}

In [29]:
frozenset.__dict__

mappingproxy({'__new__': <function frozenset.__new__(*args, **kwargs)>,
              '__repr__': <slot wrapper '__repr__' of 'frozenset' objects>,
              '__hash__': <slot wrapper '__hash__' of 'frozenset' objects>,
              '__lt__': <slot wrapper '__lt__' of 'frozenset' objects>,
              '__le__': <slot wrapper '__le__' of 'frozenset' objects>,
              '__eq__': <slot wrapper '__eq__' of 'frozenset' objects>,
              '__ne__': <slot wrapper '__ne__' of 'frozenset' objects>,
              '__gt__': <slot wrapper '__gt__' of 'frozenset' objects>,
              '__ge__': <slot wrapper '__ge__' of 'frozenset' objects>,
              '__iter__': <slot wrapper '__iter__' of 'frozenset' objects>,
              '__sub__': <slot wrapper '__sub__' of 'frozenset' objects>,
              '__rsub__': <slot wrapper '__rsub__' of 'frozenset' objects>,
              '__and__': <slot wrapper '__and__' of 'frozenset' objects>,
              '__rand__': <slot wrapper '__r

In [30]:
set.__dict__

mappingproxy({'__new__': <function set.__new__(*args, **kwargs)>,
              '__repr__': <slot wrapper '__repr__' of 'set' objects>,
              '__hash__': None,
              '__lt__': <slot wrapper '__lt__' of 'set' objects>,
              '__le__': <slot wrapper '__le__' of 'set' objects>,
              '__eq__': <slot wrapper '__eq__' of 'set' objects>,
              '__ne__': <slot wrapper '__ne__' of 'set' objects>,
              '__gt__': <slot wrapper '__gt__' of 'set' objects>,
              '__ge__': <slot wrapper '__ge__' of 'set' objects>,
              '__iter__': <slot wrapper '__iter__' of 'set' objects>,
              '__init__': <slot wrapper '__init__' of 'set' objects>,
              '__sub__': <slot wrapper '__sub__' of 'set' objects>,
              '__rsub__': <slot wrapper '__rsub__' of 'set' objects>,
              '__and__': <slot wrapper '__and__' of 'set' objects>,
              '__rand__': <slot wrapper '__rand__' of 'set' objects>,
              '__xor

In [32]:
globals()['m'] = 'появилась без присванивания'
print(m)
m = 'появилась c присваниванием'
print(m)

появилась без присванивания
появилась c присваниванием


In [33]:
import builtins
print(builtins.len is len)

True


In [36]:
list = [1, 2, 3]
try:
    list('abc')
except TypeError as e:
    print(e)

del list
list('abc')

'list' object is not callable


['a', 'b', 'c']

In [ ]:
# LEGB
# Local -> Enclosing -> Global -> Built-in

In [52]:
x = 'global'

def outer():
    x = 'enclosing'

    def inner():
        # x = 'local'
        print(x)
    
    inner()

outer()
        

enclosing


In [69]:
def make_counter(count: int = 0):
    if not count:
        count = 0

    def step():
        nonlocal count
        count = count + 1  # count += 1
        return count

    return step


count = 0
def make_counter_global():
    def step():
        global count
        count = count + 1  # count += 1
        return count

    return step

In [70]:
c1 = make_counter()
c2 = make_counter()
print(c1(), c1(), c1())
print(c2(), c2(), c2())
print()
c1 = make_counter_global()
c2 = make_counter_global()
print(c1(), c1(), c1())
print(c2(), c2(), c2())

1 2 3
1 2 3

1 2 3
4 5 6


In [76]:
# obj.attr = v
# obj.__dict__['attr'] = v

In [79]:
class Dog:
    species = 'metis'

    def __init__(self, name):
        self.name = name

    def bark(self):
        return f'{self.name}: Гав!'

iriska = Dog('Iriska')
print(iriska.__dict__)
print(vars(iriska) is iriska.__dict__)

{'name': 'Iriska'}
{'name': 'Iriska'}


In [87]:
iriska.__dict__['age'] = 1
print(iriska.age)
print(iriska.__dict__)

1
{'name': 'Iriska', 'age': 1}


In [84]:
print(type(Dog.__dict__))

<class 'mappingproxy'>


# Самостоятельно

## Блок 3 (дополнение). Две ловушки областей видимости

На лекции мы разобрали LEGB, `global`, `nonlocal` и замыкания. Здесь две ситуации, на которых чаще всего ошибаются
даже опытные разработчики. Запускайте ячейки и пробуйте предсказать вывод до запуска.

### 3.3 `UnboundLocalError`

Посмотрите на код ниже. Переменная `counter` есть в глобальном пространстве имен, по правилу LEGB функция должна ее найти.
Но код падает.

In [ ]:
counter = 0


def increment():
    counter += 1


try:
    increment()
except UnboundLocalError as e:
    print('UnboundLocalError:', e)

UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value


**Почему так происходит.** Решение, является ли имя локальным, Python принимает **не во время выполнения**, а заранее,
когда компилирует тело функции в байт-код. Правило простое: если внутри функции есть **присваивание** имени
(`x = ...`, `x += ...`, `for x in ...`, `import x`, `def x`, `class x`, `del x`), то это имя локальное **во всей функции**,
даже в строках, которые стоят до присваивания.

Запись `counter += 1` означает `counter = counter + 1`. Это присваивание, поэтому `counter` локальная.
Справа Python пытается прочитать локальную `counter`, а значения у нее еще нет: отсюда ошибка.

Список локальных имен компилятор записывает в атрибут `__code__.co_varnames`, его можно посмотреть:

In [ ]:
print(increment.__code__.co_varnames)

('counter',)


Самое коварное: присваивание может стоять **ниже** и даже в ветке, которая никогда не выполнится.
Функция ломается целиком, хотя строка с ошибкой выглядит невинно.

In [ ]:
total = 10


def report(debug=False):
    print('total =', total)     # падает здесь...
    if debug:
        total = 0               # ...из-за этой строки, хотя она не выполняется


try:
    report()
except UnboundLocalError as e:
    print('UnboundLocalError:', e)

UnboundLocalError: cannot access local variable 'total' where it is not associated with a value


**Как исправить.** Три варианта, от лучшего к худшему:

1. **Передать значение параметром и вернуть результат.** Функция не зависит от глобального состояния,
   ее легко тестировать. Это предпочтительный способ.
2. `nonlocal`, если переменная живет в объемлющей функции (замыкание).
3. `global`, если переменная в модуле. Работает, но обычно это признак плохого дизайна.

In [ ]:
def increment(value):
    return value + 1


counter = 0
counter = increment(counter)
counter = increment(counter)
print(counter)

2


**Важная тонкость: изменение объекта не является присваиванием.** Если мы не переприсваиваем имя, а меняем объект,
на который оно ссылается (`append`, `d[key] = ...`, `obj.attr = ...`), то имя остается нелокальным,
и `nonlocal`/`global` не нужны. Это прямое следствие модели «имя это ссылка на объект» из лекции 2.

In [ ]:
def make_logger():
    messages = []          # список в объемлющей функции
    count = 0

    def log(text):
        nonlocal count     # count переприсваивается, nonlocal нужен
        count += 1
        messages.append(text)   # messages только изменяется, nonlocal не нужен
        return count, messages

    return log


log = make_logger()
log('start')
print(log('stop'))

(2, ['start', 'stop'])


### 3.5 Позднее связывание в замыканиях

Попробуйте предсказать вывод, прежде чем запускать:

In [ ]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])

[2, 2, 2]


Многие ожидают `[0, 1, 2]`, а получают `[2, 2, 2]`.

**Почему.** Замыкание хранит **не значение** переменной в момент создания функции, а **ссылку на саму переменную**
(на «ячейку» объемлющей области). Значение читается в момент **вызова** функции. Все три лямбды ссылаются
на одну и ту же переменную `i`, а вызываем мы их после цикла, когда `i` уже равно 2.
Это называется **поздним связыванием** (late binding).

Проверим: у всех трех функций одна и та же ячейка.

In [ ]:
def make_funcs():
    result = []
    for i in range(3):
        def f():
            return i
        result.append(f)
    return result


funcs = make_funcs()
print([f() for f in funcs])
print([id(f.__closure__[0]) for f in funcs])     # одинаковые id
print(funcs[0].__closure__[0].cell_contents)

[2, 2, 2]
[139868363280768, 139868363280768, 139868363280768]
2


Ловушка встречается не только в учебных примерах. Типичный реальный случай: создание обработчиков в цикле
(кнопки в интерфейсе, колбэки, задачи для планировщика). Все обработчики в итоге работают с последним значением.

**Как исправить.** Нужно, чтобы каждая функция получила **свою** переменную со своим значением.

1. **Значение по умолчанию.** Оно вычисляется в момент *определения* функции (та же особенность, из-за которой был баг
   с `bucket=[]`, только здесь она нам помогает). Минус: в сигнатуре появляется лишний параметр, который можно
   случайно передать.
2. **Фабрика функций.** Каждый вызов фабрики создает новую область видимости со своей переменной. Самый чистый вариант.
3. **`functools.partial`** фиксирует аргументы прямо сейчас.

In [ ]:
from functools import partial

# 1. значение по умолчанию
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])


# 2. фабрика
def make_multiplier(k):
    def multiply(x):
        return x * k
    return multiply


multipliers = [make_multiplier(k) for k in range(3)]
print([m(10) for m in multipliers])


# 3. partial
def power(x, n):
    return x ** n


powers = [partial(power, n=n) for n in range(3)]
print([p(2) for p in powers])

[0, 1, 2]
[0, 10, 20]
[1, 2, 4]


## Блок 4. Атрибуты объектов (теория для самостоятельного изучения)

Этот блок мы не успели разобрать на лекции. Прочитайте текст и **запускайте каждую ячейку сами**.
Перед запуском попробуйте предсказать вывод: так материал запоминается гораздо лучше.

Главная мысль блока: атрибуты объекта хранятся в обычном словаре. Все, что мы узнали про `dict` и namespace'ы,
напрямую объясняет, как работают объекты в Python.

План:
1. `__dict__` экземпляра и класса
2. Порядок поиска атрибута
3. Баг: изменяемый атрибут класса
4. `getattr`, `setattr`, `hasattr`, `delattr`
5. У модулей и функций тоже есть `__dict__`
6. `__slots__`: объект без `__dict__`
7. `__getattr__`: последний шанс найти атрибут

### 4.1 `__dict__` экземпляра и класса

У каждого объекта есть собственное пространство имен: словарь `obj.__dict__`. Когда мы пишем

```python
obj.attr = value
```

Python по сути делает

```python
obj.__dict__['attr'] = value
```

Функция `vars(obj)` возвращает тот же самый словарь, просто читается приятнее.

Важно различать **два** словаря:
- `iriska.__dict__` хранит данные **конкретного экземпляра** (то, что мы присвоили через `self.name = ...`);
- `Dog.__dict__` хранит содержимое **класса**: методы и атрибуты класса, общие для всех экземпляров.

Обратите внимание: метода `bark` нет в словаре экземпляра. Методы живут в классе, а не копируются в каждый объект.

In [ ]:
class Dog:
    species = 'metis'          # атрибут класса: один на всех

    def __init__(self, name):
        self.name = name       # атрибут экземпляра: у каждого свой

    def bark(self):
        return f'{self.name}: Гав!'


iriska = Dog('Iriska')
print(iriska.__dict__)
print(vars(iriska) is iriska.__dict__)

iriska.__dict__['age'] = 1     # то же самое, что iriska.age = 1
print(iriska.age)

{'name': 'Iriska'}
True
1


Словарь класса имеет тип `mappingproxy`. Это обертка **только для чтения**: напрямую записать в нее нельзя.
Так сделано потому, что при изменении класса Python должен обновить внутренние кэши поиска атрибутов,
а прямая запись в словарь их бы обошла. Менять класс можно обычным присваиванием: `Dog.species = '...'`.

In [ ]:
print(type(Dog.__dict__))
print([k for k in Dog.__dict__ if not k.startswith('__')])

try:
    Dog.__dict__['species'] = 'husky'
except TypeError as e:
    print('TypeError:', e)

print('bark' in iriska.__dict__, 'bark' in Dog.__dict__)

<class 'mappingproxy'>
['species', 'bark']
TypeError: 'mappingproxy' object does not support item assignment
False True


### 4.2 Порядок поиска атрибута

Когда мы пишем `iriska.species`, Python ищет имя по цепочке (очень похоже на LEGB для переменных):

1. словарь экземпляра `iriska.__dict__`;
2. словарь класса `Dog.__dict__`;
3. словари родительских классов в порядке **MRO** (Method Resolution Order), его можно посмотреть в `Dog.__mro__`;
4. если нигде не нашлось, будет `AttributeError`.

Ключевая асимметрия, которую нужно запомнить:

- **чтение** `obj.attr` идет по всей цепочке: экземпляр, класс, родители;
- **присваивание** `obj.attr = ...` **всегда** пишет в словарь экземпляра.

Поэтому присваивание через экземпляр не меняет атрибут класса, а создает у экземпляра **свой** атрибут
с тем же именем, который **затеняет** атрибут класса. Другие экземпляры этого не замечают.

In [ ]:
print(iriska.species, 'species' in iriska.__dict__)
print(Dog.__mro__)

metis False
(<class '__main__.Dog'>, <class 'object'>)


In [ ]:
iriska.species = 'robot'       # создали атрибут в экземпляре
print(iriska.__dict__)

bim = Dog('Bim')
print(iriska.species, bim.species, Dog.species)

{'name': 'Iriska', 'age': 1, 'species': 'robot'}
robot metis metis


In [ ]:
del iriska.species             # удалили атрибут экземпляра...
print(iriska.species)          # ...и снова видим атрибут класса

metis


А вот если изменить атрибут **через класс**, изменение увидят все экземпляры, у которых нет своего атрибута
с таким именем:

In [ ]:
Dog.species = 'husky'
print(iriska.species, bim.species)

husky husky


### 4.3 Баг: изменяемый атрибут класса

Это близкий родственник бага с изменяемым аргументом по умолчанию `bucket=[]` из лекции 2.

Атрибут класса создается **один раз**, при выполнении тела класса. Если это список, то он один на все экземпляры.
Вызов `self.tricks.append(...)` **не является присваиванием**: это чтение атрибута `self.tricks`
(Python находит его в классе) и изменение найденного объекта. Поэтому новый трюк появляется у всех собак.

In [ ]:
class Dog:
    tricks = []                     # ОДИН список на всех собак

    def __init__(self, name):
        self.name = name

    def add_trick(self, trick):
        self.tricks.append(trick)


rex, bim = Dog('Rex'), Dog('Bim')
rex.add_trick('сидеть')
print(bim.tricks)
print(rex.__dict__)                 # у экземпляра нет своего tricks
print(rex.tricks is Dog.tricks)

['сидеть']
{'name': 'Rex'}
True


**Как правильно.** Изменяемое состояние, которое должно быть у каждого объекта своим, создаем в `__init__`.
Тогда при создании каждого экземпляра будет создаваться новый список.

Атрибуты класса подходят для **неизменяемых** общих значений: констант, настроек по умолчанию, счетчиков,
которые меняются явным присваиванием через класс (`Dog.count += 1`).

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name
        self.tricks = []            # у каждого экземпляра свой список

    def add_trick(self, trick):
        self.tricks.append(trick)


rex, bim = Dog('Rex'), Dog('Bim')
rex.add_trick('сидеть')
print(rex.tricks, bim.tricks)

['сидеть'] []


### 4.4 Динамический доступ: `getattr`, `setattr`, `hasattr`, `delattr`

Раз атрибуты лежат в словаре, к ним можно обращаться по имени, записанному в строке:

| Функция | Эквивалент |
|---|---|
| `getattr(obj, 'x')` | `obj.x` |
| `getattr(obj, 'x', default)` | `obj.x`, а если атрибута нет, то `default` (без `AttributeError`) |
| `setattr(obj, 'x', v)` | `obj.x = v` |
| `hasattr(obj, 'x')` | есть ли атрибут (проверяет всю цепочку поиска) |
| `delattr(obj, 'x')` | `del obj.x` |

Типичный случай: имя атрибута становится известно только во время выполнения программы.
Например, нужно загрузить настройки из словаря (JSON, конфиг-файла) в объект.

In [ ]:
class Settings:
    debug = False


raw = {'debug': True, 'host': 'localhost', 'port': 8080}
s = Settings()
for key, value in raw.items():
    setattr(s, key, value)

print(vars(s))
print(getattr(s, 'port'))
print(getattr(s, 'timeout', 30))
print(hasattr(s, 'timeout'))

{'debug': True, 'host': 'localhost', 'port': 8080}
8080
30
False


Еще один пример: выбор метода по имени. Так часто устроены обработчики команд.

In [ ]:
class Calculator:
    def add(self, a, b):
        return a + b

    def mul(self, a, b):
        return a * b


calc = Calculator()
for command in ['add', 'mul', 'pow']:
    method = getattr(calc, command, None)
    if method is None:
        print(f'{command}: неизвестная команда')
    else:
        print(f'{command}: {method(3, 4)}')

add: 7
mul: 12
pow: неизвестная команда


**Предостережение.** Не злоупотребляйте динамическим доступом. Код `getattr(obj, name)` сложнее читать,
а линтеры, автодополнение в IDE и `mypy` не могут его проверить. Если имя атрибута известно заранее,
пишите просто `obj.name`.

### 4.5 У модулей и функций тоже есть `__dict__`

Теперь круг замыкается. В Python модуль и функция тоже объекты, и у них тоже есть `__dict__`.

- `import math` создает объект-модуль. Запись `math.pi` означает поиск `'pi'` в `math.__dict__`.
- Глобальные переменные нашего ноутбука (или скрипта) хранятся в `__dict__` модуля `__main__`.
  Это тот же самый словарь, что возвращает `globals()`.
- У функции можно хранить собственные атрибуты, например счетчик вызовов.

In [ ]:
import math
print(math.__dict__['pi'] is math.pi)

import __main__
print(__main__.__dict__ is globals())

True
True


In [ ]:
def greet():
    greet.calls += 1
    return 'hi'


greet.calls = 0
greet()
greet()
print(greet.calls, greet.__dict__)

2 {'calls': 2}


### 4.6 `__slots__`: объект без `__dict__`

Словарь у каждого экземпляра дает гибкость (можно добавить любой атрибут в любой момент),
но стоит памяти. Если объектов очень много (миллионы точек, записей, событий), это становится заметно.

Объявление `__slots__` в классе говорит Python: у экземпляров будут **только** перечисленные атрибуты.
Тогда:
- значения хранятся в фиксированных ячейках внутри объекта, `__dict__` не создается;
- объект занимает меньше памяти, доступ к атрибутам немного быстрее;
- добавить атрибут, которого нет в `__slots__`, нельзя: будет `AttributeError`.

In [ ]:
class PointDict:
    def __init__(self, x, y):
        self.x, self.y = x, y


class PointSlots:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x, self.y = x, y


p = PointSlots(1, 2)
print(p.x, p.y)
print(hasattr(p, '__dict__'))

try:
    p.z = 3
except AttributeError as e:
    print('AttributeError:', e)

1 2
False
AttributeError: 'PointSlots' object has no attribute 'z' and no __dict__ for setting new attributes


Измерим память на 100 000 объектов с помощью модуля `tracemalloc`: он считает, сколько памяти выделил Python
между `start()` и `stop()`. Точные числа зависят от версии Python и компьютера, но разница будет видна.

In [ ]:
import tracemalloc


def measure(cls, n=100_000):
    tracemalloc.start()
    objs = [cls(i, i) for i in range(n)]
    size, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return size / 1024 / 1024


print(f'с __dict__:  {measure(PointDict):.1f} МБ')
print(f'с __slots__: {measure(PointSlots):.1f} МБ')

с __dict__:  12.2 МБ
с __slots__: 8.4 МБ


Когда использовать `__slots__`: для небольших классов-«записей», которых создается очень много.
Для обычных классов это преждевременная оптимизация, гибкость `__dict__` обычно важнее.

### 4.7 `__getattr__`: последний шанс найти атрибут

Если обычный поиск (экземпляр, класс, родители) ничего не нашел, Python проверяет, есть ли у класса метод
`__getattr__`. Если есть, вызывает его с именем атрибута. Это «последний шанс» вернуть значение вместо ошибки.

Важно: `__getattr__` вызывается **только для отсутствующих** атрибутов. Существующие атрибуты находятся
обычным путем, и метод даже не вызывается.

Пример: объект-обертка над словарем, у которого ключи доступны как атрибуты.

In [ ]:
class Record:
    def __init__(self, data):
        self._data = data

    def __getattr__(self, name):
        print(f'  __getattr__({name!r})')
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(name) from None


r = Record({'title': 'Dune', 'year': 1965})
print(r.title)
print(r.year)
print(r._data)                          # нашелся обычным путем, __getattr__ не вызван
print(getattr(r, 'author', 'неизвестен'))

  __getattr__('title')
Dune
  __getattr__('year')
1965
{'title': 'Dune', 'year': 1965}
  __getattr__('author')
неизвестен


Обратите внимание на `raise AttributeError` в конце. Если атрибута нет, `__getattr__` обязан бросать именно
`AttributeError`, а не `KeyError`: тогда корректно работают `hasattr` и `getattr` со значением по умолчанию.

Подробнее про управление атрибутами (`@property`, `__setattr__`, дескрипторы) поговорим на лекции 8.

### Итоги блока 4

- Атрибуты экземпляра хранятся в `obj.__dict__`, атрибуты и методы класса в `Cls.__dict__` (`mappingproxy`).
- Чтение атрибута идет по цепочке: экземпляр, класс, родители по MRO.
- Присваивание `obj.x = ...` всегда пишет в экземпляр и может затенить атрибут класса.
- Изменяемые значения (списки, словари) не храним в атрибутах класса, создаем их в `__init__`.
- `getattr` / `setattr` / `hasattr` / `delattr` дают доступ к атрибутам по имени-строке.
- Модули и функции тоже объекты со своим `__dict__`; `globals()` это `__dict__` текущего модуля.
- `__slots__` убирает `__dict__` и экономит память, но запрещает новые атрибуты.
- `__getattr__` вызывается только когда атрибут не найден обычным путем.

### Вопросы для самопроверки

1. Почему метода `bark` нет в `iriska.__dict__`, но вызов `iriska.bark()` работает?
2. Что выведет код ниже и почему?

```python
class A:
    x = 1

a, b = A(), A()
a.x = 2
A.x = 3
print(a.x, b.x)
```

3. Чем отличаются `self.items.append(1)` и `self.items = [1]`, если `items` объявлен как атрибут класса?
4. Почему у объекта класса со `__slots__` нельзя создать новый атрибут?
5. Когда вызывается `__getattr__`, а когда нет?

# Домашнее задание к лекции 4

Все файлы кладите в папку `hw4/`. В каждом `.py` файле должен быть блок `if __name__ == '__main__':`
с проверками (`assert`), который запускается без ошибок. Приведенные проверки минимальные: добавьте свои,
особенно на граничные случаи.

Задания 1-4 обязательные. Задания и пункты со звездочкой (*) дополнительные, они сложнее и пригодятся тем,
кто идет на автомат.

Запрещено использовать готовые решения из стандартной библиотеки там, где задача их реализовать
(`functools.lru_cache`, `functools.cache`, `collections.OrderedDict`, `dict`/`set` внутри `MyDict`).

#### Задание 1. Своя хеш-таблица

В файле `hw4/my_dict.py` реализуйте класс `MyDict`: упрощенный словарь на основе хеш-таблицы с цепочками.
Внутри класса нельзя использовать `dict` и `set`, только списки и кортежи.

Требования:
- корзины хранятся в списке списков, номер корзины: `hash(key) % len(self._buckets)`;
- при коллизии пары `(key, value)` лежат в одной корзине, нужный ключ ищется через `==`;
- операции: `d[key] = value`, `d[key]`, `del d[key]`, `key in d`, `len(d)`, `d.get(key, default=None)`;
- отсутствующий ключ в `d[key]` и `del d[key]` вызывает `KeyError`;
- когда число элементов становится больше 2/3 от числа корзин, число корзин увеличивается в 2 раза и все пары
  раскладываются заново;
- метод `stats()` возвращает кортеж `(число корзин, число элементов, длина самой длинной корзины)`;
- `__repr__` выводит содержимое в виде `MyDict({'a': 1, 'b': 2})`.

```python
class MyDict:
    def __init__(self, size=8):
        self._buckets = [[] for _ in range(size)]
        self._len = 0

    # ваш код


if __name__ == '__main__':
    d = MyDict()
    for i in range(1000):
        d[f'key{i}'] = i
    assert len(d) == 1000
    assert d['key42'] == 42
    assert 'key7' in d and 'nope' not in d
    d['key42'] = -1
    assert d['key42'] == -1 and len(d) == 1000
    del d['key42']
    assert 'key42' not in d and len(d) == 999
    assert d.get('nope') is None and d.get('nope', 0) == 0
    for bad in (lambda: d['nope'], lambda: d.__delitem__('nope')):
        try:
            bad()
            assert False, 'ожидали KeyError'
        except KeyError:
            pass
    buckets, size, longest = d.stats()
    assert size == 999 and size <= buckets * 2 / 3
    print(d.stats())
```

Ответьте в комментарии в конце файла (2-3 предложения на вопрос):
1. Почему при увеличении таблицы пары нельзя просто скопировать в корзины с теми же номерами?
2. Что показывает `stats()` для 1000 строковых ключей и что он покажет, если ключами будут объекты,
   у которых `__hash__` всегда возвращает 42? Проверьте экспериментом, замерьте время вставки 1000 таких ключей.

**1\*.** Добавьте `keys()`, `values()`, `items()` и `__iter__`, которые возвращают элементы **в порядке вставки**,
как настоящий `dict`. Порядок должен сохраняться после увеличения таблицы и корректно работать после удаления.
Подсказка: вспомните схему компактного словаря с лекции (массив индексов + плотный массив записей).

#### Задание 2. Хешируемый класс-значение

В файле `hw4/money.py` напишите класс `Money` с полями `amount` (целое число копеек) и `currency` (строка, например `'RUB'`).

Требования:
- два объекта равны, если совпадают сумма и валюта; сравнение с объектом другого типа возвращает `NotImplemented`;
- объекты можно использовать как ключи словаря и элементы множества, контракт `a == b  =>  hash(a) == hash(b)` выполняется;
- после создания поля изменить нельзя: `m.amount = 5` и `m.extra = 1` вызывают `AttributeError`
  (подсказка: `__slots__` и `__setattr__`; внутри `__init__` можно использовать `object.__setattr__`);
- `m1 + m2` складывает суммы одной валюты и возвращает **новый** объект; для разных валют бросает `ValueError`;
- `__repr__` возвращает `Money(1500, 'RUB')`.

Также напишите функцию `total_by_currency(payments)`, которая принимает список `Money` и возвращает словарь
`{валюта: Money}` с суммой по каждой валюте.

```python
if __name__ == '__main__':
    assert len({Money(100, 'RUB'), Money(100, 'RUB'), Money(100, 'USD')}) == 2
    prices = {Money(100, 'RUB'): 'сто рублей'}
    assert prices[Money(100, 'RUB')] == 'сто рублей'
    assert Money(100, 'RUB') + Money(50, 'RUB') == Money(150, 'RUB')
    for action in (lambda m: setattr(m, 'amount', 5), lambda m: setattr(m, 'extra', 1)):
        try:
            action(Money(1, 'RUB'))
            assert False, 'ожидали AttributeError'
        except AttributeError:
            pass
    try:
        Money(1, 'RUB') + Money(1, 'USD')
        assert False, 'ожидали ValueError'
    except ValueError:
        pass
    pays = [Money(100, 'RUB'), Money(5, 'USD'), Money(250, 'RUB')]
    assert total_by_currency(pays) == {'RUB': Money(350, 'RUB'), 'USD': Money(5, 'USD')}
```

В комментарии (3-4 предложения) объясните, что сломалось бы, если бы поля можно было менять после того,
как объект положили в множество или использовали как ключ. Сошлитесь на пример с `Point` из лекции.

#### Задание 3. Поисковый индекс

В файле `hw4/search_index.py` реализуйте класс `SearchIndex`: простую поисковую систему по текстовым документам
на основе **инвертированного индекса** (словарь «слово -> множество id документов»). Так устроен поиск
в поисковиках, IDE и базах данных.

Требования:
- `add(doc_id, text)` добавляет документ. Слова приводятся к нижнему регистру, разделители и знаки препинания
  отбрасываются (используйте `re.findall(r'\w+', text.lower())`). Повторное добавление того же `doc_id`
  заменяет старый текст;
- `search_all(*words)` возвращает множество id документов, в которых есть **все** слова;
- `search_any(*words)` возвращает множество id документов, в которых есть **хотя бы одно** слово;
- `remove(doc_id)` удаляет документ из индекса; слово, которое больше не встречается ни в одном документе,
  должно исчезнуть из индекса полностью;
- `top_words(n)` возвращает `n` самых частых слов по всем документам в виде списка пар `(слово, количество)`;
- поиск по словам не должен перебирать все документы: сложность `search_*` зависит от числа слов в запросе
  и размера найденных множеств, а не от общего числа документов.

```python
if __name__ == '__main__':
    idx = SearchIndex()
    idx.add(1, 'Python is a programming language.')
    idx.add(2, 'A dict is a hash table; a set is a hash table too!')
    idx.add(3, 'Python dict and set are fast.')
    assert idx.search_all('python') == {1, 3}
    assert idx.search_all('dict', 'set') == {2, 3}
    assert idx.search_all('python', 'hash') == set()
    assert idx.search_any('language', 'fast') == {1, 3}
    assert idx.search_all('PYTHON') == {1, 3}
    assert idx.search_all('nothing') == set()
    assert idx.top_words(2) == [('a', 5), ('is', 3)]
    idx.remove(1)
    assert idx.search_all('python') == {3}
    assert idx.search_any('language', 'programming') == set()
    idx.add(3, 'Totally new text')
    assert idx.search_all('python') == set()
    assert idx.search_all('new') == {3}
```

Подумайте, какие структуры данных хранить, чтобы `remove` и повторный `add` работали без перебора всего индекса.
Выбор коротко обоснуйте в docstring класса.

**3\*.** Добавьте `search_ranked(query, limit=10)`: принимает строку запроса и возвращает список id документов,
отсортированных по релевантности. Релевантность документа: сумма количеств вхождений слов запроса в документе.
Документы без единого слова запроса в результат не попадают; при равной релевантности первым идет меньший id.

#### Задание 4. Кэширование через замыкание

В файле `hw4/memoize.py` напишите функцию `memoize(func)`, которая возвращает новую функцию с кэшем результатов.
Используйте замыкание: кэш (словарь) живет в объемлющей функции, а не в глобальной переменной.

Требования:
- ключ кэша строится из позиционных и именованных аргументов, `f(1, b=2)` и `f(1, b=3)` кэшируются отдельно;
- если аргументы нехешируемые (например, список), функция вызывается без кэша и не падает;
- у возвращаемой функции есть атрибуты `hits` и `misses` (число попаданий и промахов кэша)
  и метод `cache_clear()`, который очищает кэш и обнуляет счетчики (вспомните `__dict__` у функций);
- у возвращаемой функции сохраняются `__name__` и `__doc__` исходной.

```python
if __name__ == '__main__':
    calls = []

    def slow_square(x, power=2):
        '''Возводит в степень.'''
        calls.append(x)
        return x ** power

    sq = memoize(slow_square)
    assert sq(3) == 9 and sq(3) == 9 and sq(3, power=3) == 27
    assert calls == [3, 3]
    assert (sq.hits, sq.misses) == (1, 2)
    assert sq.__name__ == 'slow_square' and sq.__doc__ == 'Возводит в степень.'
    total = memoize(sum)
    assert total([1, 2, 3]) == 6 and total([1, 2, 3]) == 6   # нехешируемый аргумент: вызов без кэша
    assert (total.hits, total.misses) == (0, 0)
    sq.cache_clear()
    assert (sq.hits, sq.misses) == (0, 0)
    sq(3)
    assert calls[-1] == 3
```

**Эксперимент с рекурсией.** Есть рекурсивная функция:

```python
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
```

Сравните два варианта и для каждого выведите `misses` после вызова с `n = 30` и время работы:

```python
fast_fib = memoize(fib)     # вариант A
fib = memoize(fib)          # вариант B
```

Один из вариантов ускоряет вычисление в тысячи раз, другой почти не помогает. В комментарии объясните почему,
используя правило LEGB: какое имя ищет рекурсивный вызов `fib(n - 1)` внутри функции и в какой момент.

#### Задание 5\*. LRU-кэш на обычном словаре

В файле `hw4/lru.py` реализуйте класс `LRUCache(capacity)`: кэш ограниченного размера, который при переполнении
выбрасывает элемент, к которому **дольше всего не обращались** (Least Recently Used). Такие кэши стоят перед
базами данных, в браузерах и в `functools.lru_cache`.

Требования:
- `get(key, default=None)` и `put(key, value)` работают за O(1) в среднем;
- обращение через `get` и перезапись через `put` делают ключ «самым свежим»;
- при добавлении нового ключа сверх `capacity` удаляется самый старый ключ;
- `len(cache)`, `key in cache` (проверка **не** должна делать ключ свежим);
- используйте только обычный `dict` и то, что порядок вставки в нем гарантирован. `OrderedDict` использовать нельзя.

```python
if __name__ == '__main__':
    c = LRUCache(2)
    c.put('a', 1)
    c.put('b', 2)
    assert c.get('a') == 1        # теперь 'a' свежее, чем 'b'
    c.put('c', 3)                 # вытесняется 'b'
    assert 'b' not in c and c.get('a') == 1 and c.get('c') == 3
    c.put('a', 10)                # перезапись делает 'a' свежим
    c.put('d', 4)                 # вытесняется 'c'
    assert c.get('c') is None and c.get('a') == 10 and len(c) == 2
```

В комментарии объясните, почему удаление самого старого элемента работает за O(1)
и какую конструкцию вы для этого использовали.

#### Задание 6\*. Конфиг с доступом через точку

В файле `hw4/config.py` напишите класс `Config`, который оборачивает вложенный словарь (например, прочитанный из JSON)
и дает доступ к значениям через атрибуты, на чтение и на запись:

```python
if __name__ == '__main__':
    raw = {'db': {'host': 'localhost', 'port': 5432}, 'debug': True}
    cfg = Config(raw)
    assert cfg.debug is True
    assert cfg.db.host == 'localhost' and cfg.db.port == 5432
    assert getattr(cfg, 'timeout', 30) == 30
    assert hasattr(cfg.db, 'user') is False
    cfg.db.port = 6543                    # запись меняет исходный словарь
    assert raw['db']['port'] == 6543
    cfg.timeout = 10
    assert raw['timeout'] == 10
    assert cfg.to_dict() is raw
    assert 'db' in dir(cfg)               # подсказка: метод __dir__
```

Требования: используйте `__getattr__` и `__setattr__`, вложенные словари оборачивайте в `Config` при обращении,
для отсутствующего ключа бросайте `AttributeError`. В комментарии объясните:
1. почему `hasattr` и `getattr` со значением по умолчанию перестанут работать, если бросать `KeyError`;
2. как вы избежали бесконечной рекурсии при обращении к собственному атрибуту, в котором хранится словарь.